# Séance 10 — Modèles génératifs

**Objectifs.**
- Autoencodeur : reconstruction, notion d'espace latent (échauffement, plus simple que
  l'entraînement adversarial).
- GAN : principe de l'apprentissage adversarial (générateur vs discriminateur), à l'aide
  d'un `GANTrainer` **fourni à trous** (fichier `gan_trainer_skeleton.py`) : la boucle
  alternée générateur/discriminateur (deux optimiseurs, mises à jour non simultanées) ne
  rentre pas dans le `Trainer` habituel — et c'est précisément l'objet pédagogique de cette
  séance, pas la plomberie autour.

Dataset : Fashion-MNIST (comme séances 7-8), pour rester léger en calcul sur CPU.


In [ ]:
# !pip install -q torch torchvision matplotlib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import torchvision
from torchvision import transforms

import matplotlib.pyplot as plt

from training_toolbox import Trainer, accuracy
from gan_trainer_skeleton_corr import GANTrainer

torch.manual_seed(0)

## Partie 1 — Autoencodeur (échauffement)

Un autoencodeur apprend à reconstruire son entrée après être passé par une représentation
intermédiaire de dimension réduite (le **code**, ou représentation dans l'**espace
latent**). S'il arrive à reconstruire correctement l'image à partir d'un code beaucoup plus
petit que l'image elle-même, c'est que ce code capture l'essentiel de l'information utile.

**Astuce d'implémentation.** Le `Trainer` habituel calcule `loss_fn(preds, y)` à partir de
paires `(x, y)` fournies par le dataloader. Pour un autoencodeur, la "cible" `y` est
l'image `x` elle-même : il suffit d'un dataset qui renvoie `(image, image)` au lieu de
`(image, label)`, et le `Trainer` fonctionne alors **sans aucune modification**.


In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(torch.flatten)])

mnist_train = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
mnist_test = torchvision.datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)


class AutoencoderDataset(Dataset):
    """Enveloppe un dataset d'images pour renvoyer (image, image) au lieu de (image,
    label) : la cible de la reconstruction est l'image elle-même."""

    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, _label = self.base_dataset[idx]
        return img, img


ae_train_loader = DataLoader(AutoencoderDataset(mnist_train), batch_size=128, shuffle=True)
ae_test_loader = DataLoader(AutoencoderDataset(mnist_test), batch_size=256)


**À vous de jouer.** Complétez `Autoencoder` : un encodeur qui réduit l'image (784
pixels après aplatissement) à un code de dimension `latent_dim`, et un décodeur qui fait le
chemin inverse. On reste volontairement simple (une seule couche cachée de chaque côté) :
l'objectif est de voir le principe fonctionner, pas d'obtenir des reconstructions parfaites.

- **Encodeur** : `Linear(784, latent_dim)`
- **Décodeur** : `Linear(latent_dim, 784)` suivi d'une `Sigmoid` (les pixels sont dans
  `[0, 1]`)


In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.encoder = nn.Linear(784, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 784), nn.Sigmoid())

    def forward(self, x):
        code = self.encoder(x)
        return self.decoder(code)


autoencoder = Autoencoder(latent_dim=16)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

trainer = Trainer(autoencoder, optimizer, loss_fn)
history = trainer.fit(ae_train_loader, ae_test_loader, epochs=8)

In [ ]:
def plot_reconstruction(model, images, n=6):
    model.eval()
    with torch.no_grad():
        reconstructions = model(images[:n])
    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
    for i in range(n):
        axes[0, i].imshow(images[i].view(28, 28), cmap="gray")
        axes[0, i].axis("off")
        axes[1, i].imshow(reconstructions[i].view(28, 28), cmap="gray")
        axes[1, i].axis("off")
    axes[0, 0].set_ylabel("original")
    axes[1, 0].set_ylabel("reconstruit")
    plt.tight_layout()
    plt.show()

x_test_batch, _ = next(iter(ae_test_loader))
plot_reconstruction(autoencoder, x_test_batch)


**Questions.**

- À quoi correspond `code = autoencoder.encoder(x)` ? Que représente ce vecteur de
  dimension 16 par rapport à l'image de 784 pixels ?
- Que se passerait-il si `latent_dim` était égal à 784 (la dimension de l'image) ? Et si
  `latent_dim = 1` ?
- *(Optionnel, si le temps le permet)* Essayez d'ajouter une couche cachée supplémentaire de
  chaque côté (encodeur et décodeur), en gardant une structure "miroir". Cela améliore-t-il
  la reconstruction ?
- *(Optionnel)* Un autoencodeur entraîné à reconstruire des images propres à partir
  d'images **bruitées** en entrée (mêmes images cibles, mais `x` bruité) devient un
  débruiteur. Essayez, en ajoutant par exemple `x_noisy = x + 0.2 * torch.randn_like(x)` côté
  entrée du dataset.


## Partie 2 — GAN

Contrairement à l'autoencodeur, un GAN n'apprend pas à reconstruire une entrée : il apprend
à **générer** des images ressemblant aux données d'entraînement, à partir de bruit aléatoire,
via une compétition entre deux réseaux :

- le **générateur** transforme un vecteur de bruit `z` en image ;
- le **discriminateur** essaie de distinguer les vraies images des images générées.

Ces deux réseaux ont des objectifs opposés et sont entraînés en alternance, avec **deux
optimiseurs distincts** — c'est ce qui empêche de réutiliser le `Trainer` habituel tel quel.
Le fichier `gan_trainer_skeleton.py` fournit toute la plomberie (`fit`, historique des
pertes, génération d'échantillons) : **c'est dans ce fichier, pas dans ce notebook, que vous
devez compléter les deux méthodes `_train_discriminator_step` et `_train_generator_step`**
(voir les commentaires `TODO 1` / `TODO 2` dans le fichier).


In [ ]:
# Architectures pré-écrites (l'objet pédagogique de cette séance est la boucle
# d'entraînement, pas l'architecture générateur/discriminateur).

LATENT_DIM = 64
IMG_DIM = 28 * 28


class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, img_dim=IMG_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, img_dim), nn.Tanh(),  # sortie dans [-1, 1]
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    def __init__(self, img_dim=IMG_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(img_dim, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 1),  # logit -- pas de sigmoid, BCEWithLogitsLoss s'en charge
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
# Données à l'échelle [-1, 1] pour matcher la sortie en Tanh du générateur
gan_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
gan_train_set = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=gan_transform
)
gan_loader = DataLoader(gan_train_set, batch_size=128, shuffle=True)

generator = Generator()
discriminator = Discriminator()
opt_g = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

gan_trainer = GANTrainer(generator, discriminator, opt_g, opt_d, latent_dim=LATENT_DIM)


**Ne lancez la cellule suivante qu'une fois les `TODO` de `gan_trainer_skeleton.py`
complétés** (sinon `NotImplementedError`). Sur CPU, un nombre d'epochs raisonnable pour la
séance est de l'ordre de 15-20 : n'hésitez pas à réduire encore si c'est trop lent, et à
relancer avec plus d'epochs une fois le code validé.


In [ ]:
gan_history = gan_trainer.fit(gan_loader, epochs=15)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(gan_history["loss_d"], label="discriminateur")
ax.plot(gan_history["loss_g"], label="générateur")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
ax.set_title("Courbes de perte du GAN")
plt.show()


In [ ]:
samples = gan_trainer.generate(16)
samples = (samples + 1) / 2  # retour en [0, 1] pour l'affichage

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for ax, img in zip(axes.ravel(), samples):
    ax.imshow(img.squeeze(0).detach(), cmap="gray")
    ax.axis("off")
plt.suptitle("Échantillons générés")
plt.tight_layout()
plt.show()


**Questions de compréhension.**

- Pourquoi le générateur est-il entraîné avec le label `1` (« vraie image ») sur des images
  qu'il a lui-même générées ? Ce n'est pourtant pas la vérité...
- Dans `_train_discriminator_step`, pourquoi faut-il `detach()` (ou `torch.no_grad()`) les
  images générées, alors que dans `_train_generator_step` il ne le faut surtout pas ?
- Contrairement à la loss d'un classifieur classique, une loss de GAN qui ne décroît pas
  n'est pas forcément un mauvais signe. Pourquoi ? Que surveiller à la place (au-delà des
  courbes `loss_g`/`loss_d`) ?
- Comparez la difficulté de mise en œuvre (et la stabilité) du GAN par rapport à
  l'autoencodeur de la Partie 1. D'où vient la différence ?
- Si vous observez que toutes les images générées se ressemblent fortement les unes aux
  autres (peu de diversité) : c'est un phénomène connu, appelé *mode collapse*. À votre avis,
  pourquoi peut-il se produire dans ce type d'entraînement adversarial ?

## Pour aller plus loin (optionnel)

- GAN **conditionnel** (CGAN) : conditionner le générateur et le discriminateur sur la classe
  (embedding de label concaténé au bruit / à l'image), pour pouvoir choisir quel vêtement
  générer plutôt que de laisser le hasard décider.
- *Flow matching* : une famille de modèles génératifs plus récente que les GAN, qui évite
  l'instabilité de l'entraînement adversarial. Non traité ici (trop avancé pour ce cours),
  mais une introduction existe dans le TP `CFM_nocorr.ipynb` du cours précédent, pour qui
  veut creuser au-delà du programme.
- La séance 11 réutilisera un modèle de diffusion **pré-entraîné** (famille de modèles
  génératifs différente des GAN, egalement plus stable à l'entraînement) : l'occasion de
  comparer les échantillons obtenus.
